In [ ]:
from __future__ import annotations
import os
import re
import json
import shutil
import random
import tempfile
import subprocess
from pathlib import Path
from dataclasses import dataclass, field
from datetime import date
from typing import Callable, Optional
USE_LLM = False
SEED = 7
random.seed(SEED)
@dataclass
class Task:
    id: str
    input: str
    metadata: dict = field(default_factory=dict)
@dataclass
class Trajectory:
    task_id: str
    output: str
    steps: list = field(default_factory=list)
@dataclass
class Feedback:
    success: bool
    score: float
    detail: dict = field(default_factory=dict)
@dataclass
class Observation:
    task: Task
    trajectory: Trajectory
    feedback: Feedback
@dataclass
class StepResult:
    accepted: bool
    score: float
    note: str = ""
    intervention: str = ""
@dataclass
class EvolutionResult:
    final_score: float
    converged: bool
    base_score: float
    history: "EvolutionHistory"
    test_curve: list = field(default_factory=list)
def parse_frontmatter(text: str) -> tuple[dict, str]:
    meta: dict = {}
    if text.startswith("---"):
        end = text.find("\n---", 3)
        if end != -1:
            block = text[3:end].strip("\n")
            body = text[end + 4:].lstrip("\n")
            for line in block.splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    meta[k.strip()] = v.strip()
            return meta, body
    return meta, text
def make_frontmatter(meta: dict, body: str) -> str:
    lines = ["---"] + [f"{k}: {v}" for k, v in meta.items()] + ["---", "", body.strip(), ""]
    return "\n".join(lines)
def parse_manifest(text: str) -> dict:
    out: dict = {"entrypoint": None, "name": None, "evolvable_layers": []}
    in_layers = False
    for raw in text.splitlines():
        line = raw.rstrip()
        if not line.strip():
            continue
        if re.match(r"^\s*evolvable_layers\s*:", line):
            in_layers = True
            continue
        m = re.match(r"^\s*-\s*(\w+)\s*$", line)
        if in_layers and m:
            out["evolvable_layers"].append(m.group(1))
            continue
        in_layers = False
        m = re.search(r"entrypoint\s*:\s*(\S+)", line)
        if m:
            out["entrypoint"] = m.group(1)
        m = re.search(r"^\s*name\s*:\s*(\S+)", line)
        if m and out["name"] is None:
            out["name"] = m.group(1)
    return out

In [ ]:
class AgentWorkspace:
    def __init__(self, root: str | Path):
        self.root = Path(root)
    @property
    def manifest_path(self) -> Path: return self.root / "manifest.yaml"
    @property
    def system_prompt_path(self) -> Path: return self.root / "prompts" / "system.md"
    @property
    def fragments_dir(self) -> Path: return self.root / "prompts" / "fragments"
    @property
    def skills_dir(self) -> Path: return self.root / "skills"
    @property
    def memory_path(self) -> Path: return self.root / "memory" / "episodic.jsonl"
    def read_manifest(self) -> dict:
        return parse_manifest(self.manifest_path.read_text()) if self.manifest_path.exists() else {}
    def read_system_prompt(self) -> str:
        return self.system_prompt_path.read_text() if self.system_prompt_path.exists() else ""
    def read_skills(self) -> list[dict]:
        out = []
        if self.skills_dir.exists():
            for skill_md in sorted(self.skills_dir.glob("*/SKILL.md")):
                meta, body = parse_frontmatter(skill_md.read_text())
                meta["body"] = body
                meta.setdefault("name", skill_md.parent.name)
                out.append(meta)
        return out
    def read_fragments(self) -> list[dict]:
        out = []
        if self.fragments_dir.exists():
            for frag in sorted(self.fragments_dir.glob("*.md")):
                meta, body = parse_frontmatter(frag.read_text())
                meta["body"] = body
                meta.setdefault("name", frag.stem)
                out.append(meta)
        return out
    def read_memories(self) -> list[dict]:
        out = []
        if self.memory_path.exists():
            for line in self.memory_path.read_text().splitlines():
                line = line.strip()
                if line:
                    out.append(json.loads(line))
        return out
    def write_skill(self, name: str, description: str, handler: str, match: str, body: str):
        d = self.skills_dir / name
        d.mkdir(parents=True, exist_ok=True)
        meta = {"name": name, "description": description, "handler": handler, "match": match}
        (d / "SKILL.md").write_text(make_frontmatter(meta, body))
    def write_fragment(self, name: str, description: str, handler: str, match: str, body: str):
        self.fragments_dir.mkdir(parents=True, exist_ok=True)
        meta = {"name": name, "description": description, "handler": handler, "match": match}
        (self.fragments_dir / f"{name}.md").write_text(make_frontmatter(meta, body))
    def append_memory(self, entry: dict):
        self.memory_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.memory_path, "a") as f:
            f.write(json.dumps(entry) + "\n")
    def tree(self) -> str:
        lines = []
        def walk(p: Path, prefix=""):
            entries = sorted([e for e in p.iterdir() if e.name != ".git"],
                             key=lambda e: (e.is_file(), e.name))
            for i, e in enumerate(entries):
                last = i == len(entries) - 1
                lines.append(prefix + ("└── " if last else "├── ") + e.name +
                             ("/" if e.is_dir() else ""))
                if e.is_dir():
                    walk(e, prefix + ("    " if last else "│   "))
        lines.append(self.root.name + "/")
        walk(self.root)
        return "\n".join(lines)
class VersionControl:
    def __init__(self, root: str | Path):
        self.root = str(root)
    def _git(self, *args, check=True) -> str:
        r = subprocess.run(["git", "-C", self.root, *args],
                           capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
        return r.stdout.strip()
    def init(self):
        self._git("init", "-q")
        self._git("config", "user.email", "evolver@a-evolve.dev")
        self._git("config", "user.name", "A-Evolve")
        self._git("add", "-A")
        self._git("commit", "-q", "-m", "seed workspace")
        self._git("tag", "evo-0")
    def commit_and_tag(self, message: str, tag: str):
        self._git("add", "-A")
        self._git("commit", "-q", "-m", message)
        self._git("tag", tag)
    def rollback(self):
        self._git("reset", "-q", "--hard", "HEAD")
        self._git("clean", "-fdq")
    def log_tags(self) -> str:
        return self._git("for-each-ref", "--sort=creatordate",
                         "--format=%(refname:short)  %(contents:subject)", "refs/tags")
K_M = 0.621371
LB_KG = 2.2046226218

In [ ]:
def _find_employee(text: str, world: list[dict]) -> Optional[dict]:
    for emp in sorted(world, key=lambda e: -len(e["name"])):
        if emp["name"] in text:
            return emp
    return None
def generic_lookup(task: Task, world: list[dict]) -> str:
    q = task.input
    emp = _find_employee(q, world)
    if emp is None:
        return ""
    ql = q.lower()
    if "email" in ql:                                   return emp["email"]
    if "title" in ql or "job" in ql or "role" in ql:    return emp["title"]
    if "city" in ql:                                    return emp["city"]
    if "department" in ql or "dept" in ql:              return emp["dept"]
    return ""
def h_date_diff(task: Task, world) -> str:
    ds = re.findall(r"\d{4}-\d{2}-\d{2}", task.input)[:2]
    if len(ds) < 2:
        return ""
    (y1, m1, d1), (y2, m2, d2) = (map(int, x.split("-")) for x in ds)
    return str(abs((date(y2, m2, d2) - date(y1, m1, d1)).days))
def h_unit_convert(task: Task, world) -> str:
    m = re.search(r"convert\s+([\d.]+)\s*(km|kg|miles|pounds)", task.input, re.I)
    if not m:
        return ""
    x, unit = float(m.group(1)), m.group(2).lower()
    val = x * K_M if unit == "km" else x * LB_KG
    return f"{val:.2f}"
def h_multi_hop(task: Task, world) -> str:
    emp = _find_employee(task.input, world)
    if not emp or not emp["manager"]:
        return ""
    mgr = next((e for e in world if e["name"] == emp["manager"]), None)
    return mgr["dept"] if mgr else ""
def h_aggregate(task: Task, world) -> str:
    q = task.input
    m = re.search(r"work in ([A-Z][A-Za-z]+)", q)
    if m:
        return str(sum(1 for e in world if e["dept"] == m.group(1)))
    m = re.search(r"report to ([A-Z][A-Za-z]+ [A-Z][A-Za-z]+)", q)
    if m:
        return str(sum(1 for e in world if e["manager"] == m.group(1)))
    return ""
def h_multi_constraint(task: Task, world) -> str:
    q = task.input
    dept = re.search(r"in ([A-Z][A-Za-z]+) whose", q)
    title = re.search(r"title is ([A-Za-z ]+?)\??$", q.strip())
    if not (dept and title):
        return ""
    d, t = dept.group(1), title.group(1).strip()
    hit = [e for e in world if e["dept"] == d and e["title"] == t]
    return hit[0]["name"] if hit else ""
HANDLER_LIBRARY: dict[str, Callable[[Task, list], str]] = {
    "date_diff": h_date_diff,
    "unit_convert": h_unit_convert,
    "multi_hop": h_multi_hop,
    "aggregate": h_aggregate,
    "multi_constraint": h_multi_constraint,
}
def apply_normalizer(norm: dict, ans: str) -> str:
    if norm.get("op") == "lower" and norm.get("when_contains", "") in ans:
        return ans.lower()
    return ans
class BaseAgent:
    def __init__(self, workspace_dir: str | Path, world: list[dict]):
        self.workspace = AgentWorkspace(workspace_dir)
        self.world = world
        self.system_prompt = ""
        self.skills: list[dict] = []
        self.routes: list[tuple] = []
        self.normalizers: list[dict] = []
        self.reload_from_fs()
    def reload_from_fs(self):
        self.system_prompt = self.workspace.read_system_prompt()
        self.skills = self.workspace.read_skills()
        self.routes = []
        for card in self.skills + self.workspace.read_fragments():
            if card.get("handler") and card.get("match"):
                try:
                    self.routes.append((re.compile(card["match"], re.I), card["handler"]))
                except re.error:
                    pass
        self.normalizers = [m for m in self.workspace.read_memories()
                            if m.get("type") == "normalizer"]
    def export_to_fs(self):
        pass
    def solve(self, task: Task) -> Trajectory:
        raise NotImplementedError
class QueryAgent(BaseAgent):
    def solve(self, task: Task) -> Trajectory:
        steps = []
        handler_name = None
        for rgx, name in self.routes:
            if rgx.search(task.input):
                handler_name = name
                break
        if handler_name:
            steps.append(f"route→{handler_name}")
            raw = HANDLER_LIBRARY[handler_name](task, self.world)
        else:
            steps.append("route→generic_lookup")
            raw = generic_lookup(task, self.world)
        ans = raw
        for norm in self.normalizers:
            new = apply_normalizer(norm, ans)
            if new != ans:
                steps.append(f"normalize:{norm.get('op')}")
                ans = new
        return Trajectory(task_id=task.id, output=ans, steps=steps)

In [ ]:
EMPLOYEES = [
    ("Alice Chen",   "Engineering", "Staff Engineer",      "Dana Patel",  "2019-04-15", "Austin", "A.Chen@Acme.com"),
    ("Bob Ruiz",     "Engineering", "Software Engineer",   "Alice Chen",  "2021-07-01", "Austin", "B.Ruiz@Acme.com"),
    ("Carol Diaz",   "Sales",       "Account Executive",   "Erin Wong",   "2020-02-10", "Denver", "C.Diaz@Acme.com"),
    ("Dana Patel",   "Engineering", "Engineering Manager", "Frank Li",    "2016-09-05", "Seattle","D.Patel@Acme.com"),
    ("Erin Wong",    "Sales",       "Sales Manager",       "Frank Li",    "2017-11-20", "Denver", "E.Wong@Acme.com"),
    ("Frank Li",     "Executive",   "VP",                  None,          "2014-01-08", "Seattle","F.Li@Acme.com"),
    ("Grace Kim",    "Sales",       "Account Executive",   "Erin Wong",   "2022-03-30", "Denver", "G.Kim@Acme.com"),
    ("Henry Park",   "Engineering", "Software Engineer",   "Dana Patel",  "2023-06-12", "Austin", "H.Park@Acme.com"),
]
WORLD = [dict(zip(["name", "dept", "title", "manager", "start_date", "city", "email"], row))
         for row in EMPLOYEES]
UNIQUE_DT = [("Engineering", "Engineering Manager"), ("Sales", "Sales Manager"),
             ("Executive", "VP"), ("Engineering", "Staff Engineer")]
def _gold(category: str, q: str) -> str:
    t = Task(id="gold", input=q)
    if category == "lookup":           return generic_lookup(t, WORLD)
    if category == "email_format":     return generic_lookup(t, WORLD).lower()
    if category == "date_diff":        return h_date_diff(t, WORLD)
    if category == "unit_convert":     return h_unit_convert(t, WORLD)
    if category == "multi_hop":        return h_multi_hop(t, WORLD)
    if category == "aggregate":        return h_aggregate(t, WORLD)
    if category == "multi_constraint": return h_multi_constraint(t, WORLD)
    raise ValueError(category)
class QueryBench:
    CATEGORIES = ["lookup", "email_format", "date_diff", "unit_convert",
                  "multi_hop", "aggregate", "multi_constraint"]
    def __init__(self, seed: int = SEED):
        self.rng = random.Random(seed)
        self._cache: dict[str, list[Task]] = {}
    def _one(self, category: str, idx: int) -> Task:
        r = self.rng
        if category == "lookup":
            emp = r.choice(WORLD)
            q = r.choice([f"What is {emp['name']}'s job title?",
                          f"Which city does {emp['name']} work in?",
                          f"Which department is {emp['name']} in?"])
        elif category == "email_format":
            emp = r.choice(WORLD)
            q = f"What is {emp['name']}'s email address?"
        elif category == "date_diff":
            y = r.randint(2018, 2023)
            d1 = date(y, r.randint(1, 6), r.randint(1, 28))
            d2 = date(y + r.randint(0, 2), r.randint(7, 12), r.randint(1, 28))
            q = f"How many days are there between {d1.isoformat()} and {d2.isoformat()}?"
        elif category == "unit_convert":
            if r.random() < 0.5:
                q = f"Convert {round(r.uniform(3, 90), 1)} km to miles."
            else:
                q = f"Convert {round(r.uniform(2, 60), 1)} kg to pounds."
        elif category == "multi_hop":
            emp = r.choice([e for e in WORLD if e["manager"]])
            q = f"What department does {emp['name']}'s manager work in?"
        elif category == "aggregate":
            if r.random() < 0.5:
                q = f"How many employees work in {r.choice(['Engineering', 'Sales'])}?"
            else:
                q = f"How many employees report to {r.choice(['Erin Wong', 'Dana Patel'])}?"
        elif category == "multi_constraint":
            dept, title = r.choice(UNIQUE_DT)
            q = f"What is the name of the employee in {dept} whose title is {title}?"
        else:
            raise ValueError(category)
        return Task(id=f"{category}-{idx}", input=q,
                    metadata={"category": category, "gold": _gold(category, q)})
    def get_tasks(self, split: str) -> list[Task]:
        if split in self._cache:
            return self._cache[split]
        per = {"train": 3, "trial": 2, "test": 4}[split]
        tasks = [self._one(c, f"{split}{i}") for c in self.CATEGORIES for i in range(per)]
        self._cache[split] = tasks
        return tasks
    def evaluate(self, task: Task, trajectory: Trajectory) -> Feedback:
        gold = task.metadata["gold"]
        out = trajectory.output
        ok = (out == gold)
        return Feedback(
            success=ok,
            score=1.0 if ok else 0.0,
            detail={"gold": gold, "got": out,
                    "ci_equal": (out.lower() == gold.lower()) and not ok},
        )
class LLMProvider:
    def complete(self, prompt: str) -> str:
        raise NotImplementedError
class LocalProvider(LLMProvider):
    def complete(self, prompt: str) -> str:
        cat = prompt.split("CLUSTER=")[-1].splitlines()[0].strip() if "CLUSTER=" in prompt else "?"
        return f"ACCEPT — dominant failure cluster '{cat}'; synthesize a targeted skill."
class AnthropicProvider(LLMProvider):
    def __init__(self, model: str = "claude-sonnet-4-6"):
        from anthropic import Anthropic
        self.client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        self.model = model
    def complete(self, prompt: str) -> str:
        msg = self.client.messages.create(
            model=self.model, max_tokens=200,
            messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
def make_provider() -> LLMProvider:
    if USE_LLM and os.environ.get("ANTHROPIC_API_KEY"):
        try:
            return AnthropicProvider()
        except Exception as e:
            print(f"[provider] Anthropic unavailable ({e}); using LocalProvider.")
    return LocalProvider()

In [ ]:
class TrialRunner:
    def __init__(self, workspace_root: str | Path, bench: QueryBench, world: list[dict]):
        self.workspace_root, self.bench, self.world = str(workspace_root), bench, world
    def run(self, split: str) -> tuple[float, list[Observation]]:
        agent = QueryAgent(self.workspace_root, self.world)
        obs = []
        for task in self.bench.get_tasks(split):
            traj = agent.solve(task)
            fb = self.bench.evaluate(task, traj)
            obs.append(Observation(task, traj, fb))
        score = sum(o.feedback.score for o in obs) / len(obs)
        return score, obs
class EvolutionHistory:
    def __init__(self):
        self.trial_scores: list[float] = []
        self.versions: list[dict] = []
    def record_version(self, tag: str, intervention: str, trial_score: float):
        self.versions.append({"tag": tag, "intervention": intervention,
                              "trial_score": trial_score})
        self.trial_scores.append(trial_score)
    def egl(self, patience: int = 2, eps: float = 1e-9) -> tuple[bool, float]:
        s = self.trial_scores
        if len(s) <= patience:
            return False, float("inf")
        recent = s[-(patience + 1):]
        improvements = [recent[i + 1] - recent[i] for i in range(len(recent) - 1)]
        last = improvements[-1]
        return all(d <= eps for d in improvements), last
class Observer:
    def __init__(self, workspace_root: str | Path):
        self.dir = Path(workspace_root) / "evolution" / "observations"
        self.dir.mkdir(parents=True, exist_ok=True)
    def collect(self, cycle: int, observations: list[Observation]):
        path = self.dir / f"batch_{cycle:04d}.jsonl"
        with open(path, "w") as f:
            for o in observations:
                f.write(json.dumps({
                    "task_id": o.task.id, "input": o.task.input,
                    "output": o.trajectory.output, "success": o.feedback.success,
                    "detail": o.feedback.detail}) + "\n")
        return path
class EvolutionEngine:
    def step(self, workspace, observations, history, trial) -> StepResult:
        raise NotImplementedError
class GuidedSynthesisEngine(EvolutionEngine):
    def __init__(self, provider: Optional[LLMProvider] = None):
        self.provider = provider or make_provider()
    CLUSTER_ARTIFACT = {
        "date_diff":        ("skill",    "date_arithmetic"),
        "unit_convert":     ("skill",    "unit_conversion"),
        "multi_hop":        ("skill",    "relational_lookup"),
        "aggregate":        ("skill",    "aggregation"),
        "multi_constraint": ("fragment", "multi_constraint_filtering"),
        "normalize_lower":  ("memory",   "normalize_lower"),
    }
    @classmethod
    def _already_present(cls, workspace: AgentWorkspace, cluster: str) -> bool:
        kind, name = cls.CLUSTER_ARTIFACT[cluster]
        if kind == "skill":
            return (workspace.skills_dir / name / "SKILL.md").exists()
        if kind == "fragment":
            return (workspace.fragments_dir / f"{name}.md").exists()
        if kind == "memory":
            return any(m.get("type") == "normalizer" and m.get("op") == "lower"
                       for m in workspace.read_memories())
        return False
    @staticmethod
    def _classify(obs: Observation) -> Optional[str]:
        q, d = obs.task.input, obs.feedback.detail
        if d.get("ci_equal"):
            return "normalize_lower"
        if len(re.findall(r"\d{4}-\d{2}-\d{2}", q)) >= 2:           return "date_diff"
        if re.search(r"\bconvert\b\s+[\d.]+\s*(km|kg|miles|pounds)", q, re.I): return "unit_convert"
        if re.search(r"\bwhose title is\b", q, re.I):               return "multi_constraint"
        if re.search(r"\bmanager\b.*\b(work|department)\b", q, re.I):return "multi_hop"
        if re.search(r"\bhow many\b|\bnumber of\b", q, re.I):       return "aggregate"
        return None
    def _synthesize(self, workspace: AgentWorkspace, cluster: str, example: str):
        if cluster == "date_diff":
            workspace.write_skill(
                "date_arithmetic",
                "TRIGGER when a question contains two ISO dates and asks for a duration.",
                "date_diff", r"\b\d{4}-\d{2}-\d{2}\b.*\b\d{4}-\d{2}-\d{2}\b",
                "## Date arithmetic\nParse both YYYY-MM-DD dates and return |Δ days| as an integer.")
            return ("skill", "date_arithmetic")
        if cluster == "unit_convert":
            workspace.write_skill(
                "unit_conversion",
                "TRIGGER on 'Convert <n> <unit> to <unit>' questions.",
                "unit_convert", r"\bconvert\b\s+[\d.]+\s*(?:km|kg|miles|pounds)\b",
                "## Unit conversion\nkm→mi ×0.621371, kg→lb ×2.2046226218. Round to 2 dp.")
            return ("skill", "unit_conversion")
        if cluster == "multi_hop":
            workspace.write_skill(
                "relational_lookup",
                "TRIGGER on relational questions like '<X>'s manager's department'.",
                "multi_hop", r"\bmanager\b.*\b(?:work|department)\b",
                "## Relational lookup\nResolve the reference (person→manager) BEFORE reading the field.")
            return ("skill", "relational_lookup")
        if cluster == "aggregate":
            workspace.write_skill(
                "aggregation",
                "TRIGGER on counting questions ('how many employees …').",
                "aggregate", r"\bhow many\s+employees\b",
                "## Aggregation\nFilter the records by the stated predicate, then COUNT them.")
            return ("skill", "aggregation")
        if cluster == "multi_constraint":
            workspace.write_fragment(
                "multi_constraint_filtering",
                "TRIGGER when a question states MULTIPLE constraints ('in <D> whose title is <T>').",
                "multi_constraint", r"\bwhose title is\b",
                "## Read ALL constraints\nExtract every filter clause and apply them jointly before answering.")
            return ("fragment", "multi_constraint_filtering")
        if cluster == "normalize_lower":
            common = "@" if "@" in example else ""
            workspace.append_memory({
                "type": "normalizer", "op": "lower", "when_contains": common,
                "note": "Answers are graded case-insensitively; emit lowercase for matches."})
            return ("memory", "normalize_lower")
        raise ValueError(cluster)
    def step(self, workspace: AgentWorkspace, observations: list[Observation],
             history: EvolutionHistory, trial: TrialRunner) -> StepResult:
        clusters: dict[str, list[Observation]] = {}
        for o in observations:
            if not o.feedback.success:
                c = self._classify(o)
                if c:
                    clusters.setdefault(c, []).append(o)
        ws = workspace
        clusters = {c: v for c, v in clusters.items() if not self._already_present(ws, c)}
        if not clusters:
            return StepResult(accepted=False, score=0.0,
                              note="No NEW fixable clusters — every detected capability is "
                                   "already present. Evolution has converged.")
        cluster = max(sorted(clusters), key=lambda c: len(clusters[c]))
        members = clusters[cluster]
        example_out = members[0].feedback.detail.get("got", "")
        rationale = self.provider.complete(
            f"You are the A-Evolve curator. {len(members)} tasks failed in the same way.\n"
            f"CLUSTER={cluster}\nExample wrong output: {example_out!r}\n"
            f"Reply ACCEPT/SKIP with one sentence.")
        kind, name = self._synthesize(workspace, cluster, example_out)
        cand_score, _ = trial.run("trial")
        note = f"[{cluster}] {rationale} -> wrote {kind} '{name}'  (cluster size {len(members)})"
        return StepResult(accepted=True, score=cand_score, note=note,
                          intervention=f"{kind}:{name}")

In [ ]:
class Evolver:
    def __init__(self, agent_workspace: str | Path, benchmark: QueryBench,
                 engine: Optional[EvolutionEngine] = None, world: list[dict] = WORLD,
                 verbose: bool = True):
        self.root = str(agent_workspace)
        self.bench = benchmark
        self.world = world
        self.engine = engine or GuidedSynthesisEngine()
        self.vc = VersionControl(self.root)
        self.observer = Observer(self.root)
        self.trial = TrialRunner(self.root, benchmark, world)
        self.history = EvolutionHistory()
        self.verbose = verbose
    def _log(self, *a):
        if self.verbose:
            print(*a)
    def run(self, cycles: int = 10) -> EvolutionResult:
        self.vc.init()
        base_test, _ = self.trial.run("test")
        base_trial, _ = self.trial.run("trial")
        self.history.trial_scores.append(base_trial)
        test_curve = [(0, base_test)]
        self._log(f"\n  cycle 0 (seed)         "
                  f"trial={base_trial:5.1%}   test={base_test:5.1%}   [baseline]")
        accepted_count = 0
        converged = False
        for cycle in range(1, cycles + 1):
            train_score, train_obs = self.trial.run("train")
            self.observer.collect(cycle, train_obs)
            prev_best = self.history.trial_scores[-1]
            result = self.engine.step(self.workspace(), train_obs, self.history, self.trial)
            if not result.accepted:
                self._log(f"  cycle {cycle:<2} ─ {result.note}")
                converged = True
                break
            if result.score > prev_best + 1e-9:
                accepted_count += 1
                tag = f"evo-{accepted_count}"
                self.vc.commit_and_tag(result.note[:72], tag)
                self.history.record_version(tag, result.intervention, result.score)
                test_score, _ = self.trial.run("test")
                test_curve.append((cycle, test_score))
                self._log(f"  cycle {cycle:<2} ✓ ACCEPT {tag:<6} "
                          f"trial={result.score:5.1%}   test={test_score:5.1%}   "
                          f"{result.intervention}")
            else:
                self.vc.rollback()
                self._log(f"  cycle {cycle:<2} ✗ REJECT (gate) trial={result.score:5.1%} "
                          f"≤ best {prev_best:5.1%} — rolled back  {result.intervention}")
            done, _ = self.history.egl()
            if done:
                self._log(f"  — EGL stabilized after {tag}: holdout gains have plateaued.")
                converged = True
                break
        final_test, _ = self.trial.run("test")
        return EvolutionResult(final_score=final_test, converged=converged,
                               base_score=base_test, history=self.history,
                               test_curve=test_curve)
    def workspace(self) -> AgentWorkspace:
        return AgentWorkspace(self.root)
def write_seed_workspace(root: str | Path):
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    (root / "prompts" / "fragments").mkdir(parents=True)
    (root / "skills").mkdir(parents=True)
    (root / "memory").mkdir(parents=True)
    (root / "manifest.yaml").write_text(
        "agent:\n"
        "  name: query-agent\n"
        "  entrypoint: tutorial.QueryAgent\n"
        "evolvable_layers:\n"
        "  - prompts\n"
        "  - skills\n"
        "  - memory\n")
    (root / "prompts" / "system.md").write_text(
        "# Query Agent\n\n"
        "You answer questions about the company's employee directory.\n"
        "Look up the requested field for the named employee and return ONLY the answer.\n")
    (root / "memory" / "episodic.jsonl").write_text("")
    return AgentWorkspace(root)
def ascii_curve(curve: list[tuple[int, float]], width: int = 48, height: int = 11) -> str:
    if not curve:
        return ""
    xs = [c for c, _ in curve]
    ys = [s for _, s in curve]
    x0, x1 = min(xs), max(xs)
    grid = [[" "] * width for _ in range(height)]
    def col(x): return 0 if x1 == x0 else round((x - x0) / (x1 - x0) * (width - 1))
    def row(y): return height - 1 - round(y * (height - 1))
    pts = [(col(x), row(y)) for x, y in zip(xs, ys)]
    for (c0, r0), (c1, r1) in zip(pts, pts[1:]):
        steps = max(abs(c1 - c0), abs(r1 - r0), 1)
        for s in range(steps + 1):
            cc = round(c0 + (c1 - c0) * s / steps)
            rr = round(r0 + (r1 - r0) * s / steps)
            if 0 <= rr < height and 0 <= cc < width:
                grid[rr][cc] = "·"
    for c, r in pts:
        grid[r][c] = "●"
    out = []
    for i, line in enumerate(grid):
        label = f"{(1 - i / (height - 1)) * 100:5.0f}% │"
        out.append(label + "".join(line))
    out.append("       └" + "─" * width)
    out.append("        cycle " + str(x0) + " " * (width - 12) + "cycle " + str(x1))
    return "\n".join(out)
def per_category(root: str, bench: QueryBench, world) -> dict[str, tuple[int, int]]:
    agent = QueryAgent(root, world)
    tally: dict[str, list[int]] = {c: [0, 0] for c in bench.CATEGORIES}
    for task in bench.get_tasks("test"):
        fb = bench.evaluate(task, agent.solve(task))
        cat = task.metadata["category"]
        tally[cat][1] += 1
        tally[cat][0] += int(fb.success)
    return {c: (v[0], v[1]) for c, v in tally.items()}

In [1]:
def main():
    bench = QueryBench(seed=SEED)
    run_dir = Path(tempfile.mkdtemp(prefix="aevolve_")) / "workspace"
    print("=" * 86)
    print("  A-EVOLVE TUTORIAL — a generic agent teaches itself, with zero API keys / Docker")
    print("=" * 86)
    print(f"  workspace: {run_dir}")
    print(f"  benchmark: QueryBench  |  task families: {', '.join(bench.CATEGORIES)}")
    print(f"  splits:    train={len(bench.get_tasks('train'))}  "
          f"trial(holdout)={len(bench.get_tasks('trial'))}  "
          f"test={len(bench.get_tasks('test'))}")
    write_seed_workspace(run_dir)
    print("\n" + "-" * 86)
    print("  SEED WORKSPACE (before evolution)")
    print("-" * 86)
    print(AgentWorkspace(run_dir).tree())
    before = per_category(str(run_dir), bench, WORLD)
    print("\n" + "-" * 86)
    print("  THE EVOLUTION LOOP   (Solve → Observe → Evolve → Gate → Reload)")
    print("-" * 86)
    evolver = Evolver(run_dir, bench, engine=GuidedSynthesisEngine(), world=WORLD)
    result = evolver.run(cycles=12)
    after = per_category(str(run_dir), bench, WORLD)
    print("\n" + "-" * 86)
    print("  EVOLVED WORKSPACE (after evolution — note the new skills/, fragments/, memory)")
    print("-" * 86)
    print(AgentWorkspace(run_dir).tree())
    print("\n" + "-" * 86)
    print("  GIT AUDIT TRAIL  (every accepted mutation is a tagged commit)")
    print("-" * 86)
    print(evolver.vc.log_tags())
    print("\n" + "-" * 86)
    print("  PER-CATEGORY TEST ACCURACY  (before  →  after)")
    print("-" * 86)
    for c in bench.CATEGORIES:
        b0, bt = before[c]
        a0, at = after[c]
        bar_b = "█" * b0 + "░" * (bt - b0)
        bar_a = "█" * a0 + "░" * (at - a0)
        print(f"  {c:<16}  {b0}/{bt} {bar_b:<5}   →   {a0}/{at} {bar_a:<5}")
    print("\n" + "-" * 86)
    print("  HELD-OUT TEST SCORE vs EVOLUTION CYCLE")
    print("-" * 86)
    print(ascii_curve(result.test_curve))
    print("\n" + "=" * 86)
    print(f"  RESULT   base test score : {result.base_score:5.1%}")
    print(f"           final test score: {result.final_score:5.1%}   "
          f"(+{(result.final_score - result.base_score) * 100:.1f} points)")
    print(f"           converged       : {result.converged}   "
          f"(accepted mutations: {len(result.history.versions)})")
    print("=" * 86)
    print("\n" + "-" * 86)
    print("  BONUS — proving the GATE works: propose a DELIBERATELY BAD skill")
    print("-" * 86)
    ws = AgentWorkspace(run_dir)
    good_score, _ = evolver.trial.run("trial")
    ws.write_skill("greedy_bad_skill",
                   "TRIGGER on everything (intentionally broken).",
                   "date_diff", r".*",
                   "## Broken\nRoutes every task to the wrong handler.")
    bad_score, _ = evolver.trial.run("trial")
    print(f"  holdout BEFORE bad skill : {good_score:5.1%}")
    print(f"  holdout WITH  bad skill  : {bad_score:5.1%}   "
          f"→ gate verdict: {'ACCEPT' if bad_score > good_score else 'REJECT'}")
    evolver.vc.rollback()
    restored, _ = evolver.trial.run("trial")
    print(f"  after git rollback       : {restored:5.1%}   (workspace restored)")
    print("-" * 86)
    try:
        import matplotlib
        matplotlib.use("Agg") if not os.environ.get("DISPLAY") else None
        import matplotlib.pyplot as plt
        xs = [c for c, _ in result.test_curve]
        ys = [s * 100 for _, s in result.test_curve]
        plt.figure(figsize=(7, 4))
        plt.plot(xs, ys, marker="o")
        plt.axhline(result.base_score * 100, ls="--", color="gray", label="baseline")
        plt.title("A-Evolve: held-out test accuracy per evolution cycle")
        plt.xlabel("evolution cycle"); plt.ylabel("test accuracy (%)")
        plt.ylim(0, 105); plt.grid(alpha=0.3); plt.legend()
        out_png = Path(run_dir).parent / "a_evolve_progress.png"
        plt.savefig(out_png, dpi=120, bbox_inches="tight")
        try:
            plt.show()
        except Exception:
            pass
        print(f"\n  [plot saved to {out_png}]")
    except Exception as e:
        print(f"\n  [matplotlib unavailable: {e} — ASCII chart above is your result]")
    return result
if __name__ == "__main__":
    main()

  A-EVOLVE TUTORIAL — a generic agent teaches itself, with zero API keys / Docker
  workspace: /tmp/aevolve_0qn9k8ld/workspace
  benchmark: QueryBench  |  task families: lookup, email_format, date_diff, unit_convert, multi_hop, aggregate, multi_constraint
  splits:    train=21  trial(holdout)=14  test=28

--------------------------------------------------------------------------------------
  SEED WORKSPACE (before evolution)
--------------------------------------------------------------------------------------
workspace/
├── memory/
│   └── episodic.jsonl
├── prompts/
│   ├── fragments/
│   └── system.md
├── skills/
└── manifest.yaml

--------------------------------------------------------------------------------------
  THE EVOLUTION LOOP   (Solve → Observe → Evolve → Gate → Reload)
--------------------------------------------------------------------------------------

  cycle 0 (seed)         trial=21.4%   test=21.4%   [baseline]
  cycle 1  ✓ ACCEPT evo-1  trial=35.7%   test=35.7% 